##Retrieval-Augmented Generation

Retrieve: takes a user question → finds the few most relevant pieces/chunks of text from your document

Augument: user question + retrived information (add useful information to the question before sending it to the LLM )

Generation: LLM receives the context+question and generats the answer


##RAG into two major phase

Phase A — Preparing the document (This happens before the user asks anything)

PDF → Extract text → Split into chunks → Create embeddings → Store embeddings  


Phase B — Asking the question (This happens when the user actually asks something)

Question → Embed question → Find similar chunks → Build prompt → LLM generates answer


In [7]:
#Setup

!pip install -q google-genai pypdf # google-genai: This gives Python access to Google's Gemini APIs
                                   # pypdf: This is for working with PDF files.

from google import genai
from google.genai import types
from google.colab import userdata
import numpy as np

client=genai.Client(api_key=userdata.get('Ragproject'))


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 10.3 MB/s eta 0:00:00


In [14]:
#Upload the PDF

from google.colab import files  #Importing Colab's file-upload functionality.
upload = files.upload()  #This line initiates the file upload process.This opens the upload mechanism so you can select your PDF.

pdf_name=list(upload.keys())[0]       #upload.key()-get the uploded filename since key() is not normal list we use list(),
print("Uploded pdf: ",pdf_name)      #[0] takes the first element


Saving COVER LETTER.pdf to COVER LETTER.pdf
Uploded pdf:  COVER LETTER.pdf


In [15]:
from pypdf import PdfReader
reader = PdfReader(pdf_name)
print("No.of pages in the pdf: ",len(reader.pages))

text=""
for pg in reader.pages:
  text+=pg.extract_text() + "\n"

print("Total characters in the pdf:", len(text))

print("first few characters in the pdf:", text[:200])

No.of pages in the pdf:  1
Total characters in the pdf: 1913
first few characters in the pdf: NACHAMMAI S
Madurai, Tamil Nadu
+91 8608086565
nachammai1705@gmail.com
Linkedin: linkedin.com/in/nachammai17
27  December 2025th
Hiring Manager
Tata Group
India
Dear Hiring Team,                      


In [16]:
#chunking

def chunk_text(text,chunk_size=500,overlap=150):  #overlap helps to prevent the context across chunk boundaries
  chunk=[]
  start=0
  while(start<len(text)):
    end=start+chunk_size
    chunk.append(text[start:end])
    start=end-overlap
  return chunk

chunks=chunk_text(text)
print("Total chunks:",len(chunks))

Total chunks: 6


In [18]:
#checking overlap

print(chunks[0][-150:])
print()
print(chunks[1][:150])

er Applications (Online) at SASTRA
University and have completed a Bachelor of Science in Information Technology from
Fatima College with a CGPA of 8.

er Applications (Online) at SASTRA
University and have completed a Bachelor of Science in Information Technology from
Fatima College with a CGPA of 8.


In [25]:
#embedding

EMBEDDING_MODEL='gemini-embedding-001'
EMBEDDING_DIMENSION=768

def embed(t):
  response=client.models.embed_content(
      model=EMBEDDING_MODEL,
      contents=t,
      config=EmbedContentConfig(
          output_dimensionality=EMBEDDING_DIMENSION
      )
  )
  return(response.embeddings[0].values)

chunk_embeddings = []
for i, ch in enumerate(chunks):
    chunk_embeddings.append(np.array(embed(ch)))
    print(f"Embedded chunk {i+1}/{len(chunks)}")

chunk_embeddings = np.array(chunk_embeddings)  #turns the list of chunks into np array
print("Chunk embedding shape:", chunk_embeddings.shape)


Embedded chunk 1/6
Embedded chunk 2/6
Embedded chunk 3/6
Embedded chunk 4/6
Embedded chunk 5/6
Embedded chunk 6/6
Chunk embedding shape: (6, 768)


In [27]:
#normalization

chunk_embeddings=chunk_embeddings/np.linalg.norm(chunk_embeddings, axis = 1, keepdims=True)
print("Normalized Chunk embedding shape:", chunk_embeddings.shape)

Normalized Chunk embedding shape: (6, 768)


In [38]:
#retrive function
def retrive(question,top_k=3):
  question_vector = np.array(embed(question))
  question_vector = question_vector/np.linalg.norm(question_vector)

  score = chunk_embeddings @ question_vector

  top_result = np.argsort(score)[::-1][:top_k]
  return [(chunks[i], score[i]) for i in top_result]  # [('text...', 0.83), ('text...', 0.79), ...]

In [40]:
#test retrival
result = retrive("In which company the virrtual internship was done at?")

for chunk, score in result:
  print(f"score: {score:.2f}")
  print(chunk)
  print()


score: 0.65
iability.
Additionally, during my Virtual Internship at IBM, I worked on predictive maintenance
models for industrial machinery. My responsibilities included data preprocessing, feature
engineering, model evaluation, and cloud-based implementation using IBM Cloud and
Watsonx.ai Studio. This experience strengthened my analytical thinking and ability to apply
theoretical knowledge to real-world business problems.
Along with technical competence, I bring leadership, collaboration, and a disciplined

score: 0.60
are
development, data handling, and machine learning applications. I developed an Integrated
Medical Supply Chain Management System using Django and SQLite, where I focused on
process automation, data accuracy, and real-time dashboard reporting. I also built a Python-
based E-Book and Audiobook platform emphasizing usability, accessibility, and reliability.
Additionally, during my Virtual Internship at IBM, I worked on predictive maintenance
models for industrial machin

In [45]:
#Generate answer function

MODEL = 'gemini-3.1-flash-lite'

configuration = types.GenerateContentConfig(
    system_instruction=(
        "You are an AI assistant that helps people find information inside only this document that is provided to you. "
        "If the answer is not in the context, reply exactly: "
        "I could not find this in the document. Never make any guess with the user question."
    ),
    temperature=0.5
)

def ask(question, top_k=3):
    top = retrive(question, top_k)      # list of (chunk, score)

    # combine retrieved chunks into one big string
    content = "\n\n---------\n\n".join(chunk for chunk, score in top) #this takes the retrieved chunks and combains them into one bg string

    #THE ACTUL RAG PART
    # the augumenting part that remembers retrive-augument-generate, we've alreay retrived content now we augument question
    prompt = f"""Context from the document:
     {content}
     QUESTION: {question}
    Answer using only the context above."""

    response = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config=configuration
    )
    return response.text

print(ask("In which company was the virtual internship done?"))

IBM


In [46]:
print(ask("who is pm of india"))

I could not find this in the document.
